In [1]:
import pandas as pd
import torch
import numpy as np
import napari
import zarr
from pathlib import Path
import tifffile
from src.data import _load_split_json

# 1. Load 1 example from the validation dataset

In [2]:
dfdata = pd.read_csv("./data/dfdata.csv")
split = _load_split_json("splits/binary_split_v1_ca_waves.json")

In [ ]:
#Typed, as load_split_json loads a set and the order changes every time.
val_examples = ['Pax2_Calciumwaves_injectable anaesthetic19.0',
 'Pax2_Calciumwaves_injectable anaesthetic23.0',
 'Pax2_Calciumwaves_broken1.0',
 'Pax2_Calciumwave32.0',
 'Pax2_Calciumwaves_injectable anaesthetic11.0',
 'Pax2_Calciumwaves_ACP+isoflurane6.0',
 'Pax2_Calciumwave9.0',
 'Pax2_Calciumwaves_broken18.0',
 'Pax2_Calciumwaves_ACP+isoflurane7.0',
 'Pax2_Calciumwave45.0',
 'Pax2_Calciumwaves_broken17.0',
 'Pax2_Calciumwaves_ACP+isoflurane21.0',
 'Pax2_Calciumwave21.0',
 'Pax2_Calciumwave38.0',
 'Pax2_Calciumwaves_injectable anaesthetic5.0',
 'Pax2_Calciumwaves_ACP+isoflurane12.0',
 'Pax2_Calciumwaves_broken20.0',
 'Pax2_Calciumwaves_ACP+isoflurane4.0',
 'Pax2_Calciumwaves_injectable anaesthetic13.0',
 'Pax2_Calciumwaves_injectable anaesthetic1.0']

In [4]:
filename = Path(dfdata[dfdata['UniqueID'] == val_examples[1]]["image"].values[0].replace(r"/Volumes/Data/Users","D:/Users/Federico"))
label_filename =Path(dfdata[dfdata['UniqueID'] == val_examples[1]]["label"].values[0].replace(r"/Volumes/Data/Users","D:/Users/Federico"))
print(filename)

D:\Users\Federico\Francesca\Data\2024\2024_11_01_Pax2-GCaMP6f_injectable anaesthesia\10\processedMovies\1-jumpCorrected.ome.zarr


In [5]:
movie = zarr.open(filename, mode="r")["s0"]
label = tifffile.imread(label_filename)

In [6]:
movie.shape

(5000, 480, 1024)

In [7]:
movie2 = movie[:]

# 2. build the model and load checkpoint

In [8]:
from src import inference
from src.model import build_binary_model
from src.config import load_yaml_config
config = load_yaml_config("./config/binary_model_v2_ca_waves.yml")

model_config = config["model"]
validation_config = config["validation"]
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Cuda available")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

Cuda available


In [9]:
model = build_binary_model(
    architecture=model_config["architecture"],
    spatial_dims=model_config["spatial_dims"],
    in_channels=model_config["in_channels"],
    out_channels=model_config["out_channels"],
    channels=model_config["channels"],
    strides=model_config["strides"],
    num_res_units=model_config["num_res_units"],
).to(device)

checkpoint = torch.load("checkpoints/best_model_binary_v2_cawaves.pth")

model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

# 3. Generate predictions

In [23]:
movie3 = torch.from_numpy(movie2)[None, None].to(device)


In [24]:
predictions,probabilities = inference.predict_binary(model, movie3, device, validation_config["sw_batch_size"],validation_config["prediction_sigmoid"], 
                                                     prediction_threshold=0.1, roi_size = [256,256,256])
mask = (
    predictions[0, 0]
    .detach()
    .cpu()
    .numpy()
    .astype(bool)
)

probs = probabilities[0,0].detach().cpu().numpy()

52it [00:25,  2.01it/s]


Inference and assembly: 26.5s


In [25]:
import gc
import torch
del movie3
gc.collect()
torch.cuda.empty_cache()

In [ ]:
predictions_instance_lablels = inference.binary_to_instance_labels(mask,min_size=2000)

Closing the mask


C:\Users\Marcotti-Lab\AppData\Local\Temp\ipykernel_21456\3479090891.py:25: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  labels = remove_small_objects(labels, min_size=min_size)


# 4. Display prediction in a napari window

In [ ]:
app = napari.Viewer()

In [ ]:
app.add_image(movie2)
app.add_labels(label.astype(np.uint16))
#app.add_labels(predictions.astype(np.uint16)*2,name='Prediction')
app.add_labels(predictions_instance_lablels.astype(np.uint16),name='Labels predict')
app.add_image(probs,name='Probability map',colormap='magma')

<Image layer 'movie2' at 0x2c46918ead0>